# Module 3: Window Functions Deep Dive

**Objective**: Master window functions for banking analytics - running balances, rankings, moving averages.

## Key Concepts
- `Window.partitionBy()` - Group rows for the window
- `Window.orderBy()` - Order within each partition
- `rowsBetween()` / `rangeBetween()` - Define window frame

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, avg, count, row_number, rank, dense_rank, lag, lead, first, last, to_date
from pathlib import Path

spark = SparkSession.builder.appName("Module03-WindowFunctions").master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate()
DATA_RAW = Path("../data/raw")

In [ ]:
transactions_df = spark.read.csv(str(DATA_RAW / "transactions.csv"), header=True, inferSchema=True)
accounts_df = spark.read.csv(str(DATA_RAW / "accounts.csv"), header=True, inferSchema=True)

# Parse date
txn = transactions_df.withColumn("txn_date", to_date(col("txn_datetime")))
txn.show(5)

## 1. Running Balance per Account

In [ ]:
# Define window: partition by account, order by date
account_window = Window.partitionBy("account_id").orderBy("txn_datetime").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calculate running total
txn_running = txn.withColumn("running_total", sum("amount").over(account_window))
txn_running.select("account_id", "txn_datetime", "txn_type", "amount", "running_total").show(15)

## 2. Customer Ranking by Transaction Volume

In [ ]:
# Aggregate first
customer_txn = txn.join(accounts_df.select("account_id", "customer_id"), "account_id")
customer_volume = customer_txn.groupBy("customer_id").agg(sum("amount").alias("total_volume"), count("*").alias("txn_count"))

# Rank customers
rank_window = Window.orderBy(col("total_volume").desc())
customer_ranked = customer_volume.withColumn("volume_rank", rank().over(rank_window)).withColumn("row_num", row_number().over(rank_window))
customer_ranked.show(10)

## 3. 7-Day Moving Average

In [ ]:
# Daily aggregation first
daily_txn = txn.groupBy("account_id", "txn_date").agg(sum("amount").alias("daily_amount"))

# 7-day moving average
moving_window = Window.partitionBy("account_id").orderBy("txn_date").rowsBetween(-6, 0)
daily_with_ma = daily_txn.withColumn("moving_avg_7d", avg("daily_amount").over(moving_window))
daily_with_ma.orderBy("account_id", "txn_date").show(15)

## 4. Lag and Lead - Transaction Velocity

In [ ]:
# Previous and next transaction
velocity_window = Window.partitionBy("account_id").orderBy("txn_datetime")
txn_velocity = txn.withColumn("prev_amount", lag("amount", 1).over(velocity_window)).withColumn("next_amount", lead("amount", 1).over(velocity_window))
txn_velocity.select("account_id", "txn_datetime", "amount", "prev_amount", "next_amount").show(10)

## 5. First and Last Transaction per Account

In [ ]:
fl_window = Window.partitionBy("account_id").orderBy("txn_datetime")
fl_window_desc = Window.partitionBy("account_id").orderBy(col("txn_datetime").desc())

first_last = txn.withColumn("first_txn", first("amount").over(fl_window)).withColumn("last_txn", first("amount").over(fl_window_desc))
first_last.select("account_id", "txn_datetime", "amount", "first_txn", "last_txn").show(10)

## Practice Exercises
1. Rank branches by total transaction amount (per region)
2. Calculate month-over-month growth rate per account
3. Find the top 3 transactions per customer by amount

In [ ]:
# Exercise: Top 3 transactions per customer
# Hint: Use row_number() with partition by customer, order by amount desc, then filter
# Your code here:


In [ ]:
spark.stop()